## 1. Setup y Verificacion
Cargamos variables de entorno y preparamos los modelos que se usarán en el resto del proyecto.

In [1]:
import getpass
import os
from pathlib import Path
from google import genai
from google.genai import types
from dotenv import load_dotenv
import logging

# Cargar variables de entorno desde el archivo .env
load_dotenv()
if not os.getenv("GOOGLE_API_KEY"):
    os.environ["GOOGLE_API_KEY"] = getpass.getpass("Google API key: ")

# Configuración de modelos
GENERATION_MODEL = "gemini-2.5-flash"
EMBEDDING_MODEL = "models/gemini-embedding-001"

# Parametros modelo para Verificación
VERIFY_MAX_TOKENS = 40
VERIFY_TEMPERATURE = 0.1

# Parametros modelo para Asistente
MAX_OUTPUT_TOKENS = 1024
TEMPERATURE = 0.7

# Configuracion del pdf path
PDF_PATH = Path("../data/TENERIFE.pdf")

# Configuracion de la ruta de persistencia para Chroma
CHROMA_PATH = Path("../data/chromadb/")

# Configuramos el formato del log para que muestre la hora, el nivel de severidad y el mensaje
logger = logging.getLogger("tool_calls")
logger.setLevel(logging.INFO)
logger.propagate = False  # evita que suba al root logger y salga por pantalla
handler = logging.FileHandler("../logs/tool_calls.log", encoding="utf-8")
handler.setFormatter(logging.Formatter("%(asctime)s [%(levelname)s] %(message)s"))
logger.addHandler(handler)

logging.getLogger("httpx").setLevel(logging.WARNING)

# Configuración de la API de Google Generative AI
client = genai.Client(api_key=os.getenv("GOOGLE_API_KEY"))

# Verificar la existencia del archivo PDF
try:
    PDF_PATH.resolve(strict=True)
    print(f"El archivo {PDF_PATH} existe. Continuando con el resto de comprobaciones")
except FileNotFoundError:
    print(f"El archivo {PDF_PATH} no existe. Por favor, asegúrate de que el archivo PDF esté en la ruta correcta.")
    raise FileNotFoundError(f"El archivo {PDF_PATH} no existe. Por favor, asegúrate de que el archivo PDF esté en la ruta correcta.")

# Verificr la conexión a la API de Google Generative AI
try:
    # Realizar una solicitud de prueba a la API para verificar la conexión
    response = client.models.generate_content(
        model=GENERATION_MODEL,
        contents="Di 'conexión OK'",
        config=types.GenerateContentConfig(
            max_output_tokens=VERIFY_MAX_TOKENS,
            temperature=VERIFY_TEMPERATURE
        )
    )
    print(response.text)
except Exception as e:
    print(f"Error al conectar a la API de Google Generative AI: {e}")


El archivo ..\data\TENERIFE.pdf existe. Continuando con el resto de comprobaciones
conexión OK


## 2. Lectura del PDF

In [2]:
import pdfplumber

with pdfplumber.open(PDF_PATH) as pdf:
    # pdf.pages es una lista de páginas
    # cada página tiene un método .extract_text()
    content = []
    for i, page in enumerate(pdf.pages):
        text = page.extract_text()
        if text:
            print(f"Texto de la página {i + 1}:\n{text[0:50]}\n")
            content.append(text)
        else:
            print(f"No se pudo extraer texto de la página {i + 1}.")
    print(f"Se extrajo texto de {len(pdf.pages)} páginas del PDF.")

# Unir todo el contenido extraído en un solo string
pdf_text = "\n".join(content)

Texto de la página 1:
TENERIFE – LUGARES DE INTERÉS
SITIOS QUE VER
ZONA 

Texto de la página 2:
o Auditorio de Tenerife [vídeo - ubicación]
o Plaz

Texto de la página 3:
o Parque García Sanabria [vídeo - ubicación]
o Pla

Texto de la página 4:
o Callejear dirección Plaza del Adelantado [ubicac

Texto de la página 5:
• Parque rural de Anaga
Es un espacio natural prot

Texto de la página 6:
o Mirador Pico del Inglés (aquí tenéis una de las 

No se pudo extraer texto de la página 7.
Texto de la página 8:
o Playa de Benijo [vídeo - ubicación]
Playa de are

Texto de la página 9:
De La Orotava os recomendaría aparcar en esta expl

Texto de la página 10:
o La Playa de Los Patos
o Playa del Ancón
Nota: Pa

Texto de la página 11:
tendréis poca playa (especialmente a la del Ancón)

Texto de la página 12:
• Isla Baja (Icod de los Vinos, Garachico y Punta 

Texto de la página 13:
o Al ir hacia Buenavista, aparcar cerca de la Plaz

Texto de la página 14:
o Si vais a llegar hasta Punta de Teno, cons

## 3. Chunking y metadatos
Usamos `RecursiveCharacterTextSplitter` para crear chunks con solapamiento. Ajustamos `chunk_size` y `chunk_overlap` para el contenido.

In [3]:
from langchain_core.documents import Document
from langchain_text_splitters import RecursiveCharacterTextSplitter

# Creamos los chunks de texto para el embedding
separators=["\n\n", "\n•", "\n▪", "•", "▪", "\n", " ", ""]
chunk_size = 1000 # 1 token ≈ 4 caracteres. Entonces 100 aprox 250 tokens.
chunk_overlap = 200
def create_chunks(text, separators = separators, chunk_size=chunk_size, overlap=chunk_overlap):
    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size=chunk_size,
        chunk_overlap=overlap,
        separators=separators
    )
    chunks = text_splitter.create_documents([text])
    print(f"Se han creado {len(chunks)} chunks de texto para el embedding.")
    return chunks  

# Dividimos en chunks el texto del PDF
chunks = create_chunks(pdf_text, separators=separators, chunk_size=chunk_size, overlap=chunk_overlap)

# Comrpobamos los chunks
for i, chunk in enumerate(chunks[:3]):
    print(f"--- Chunk {i+1} ({len(chunk.page_content)} chars) ---")
    print(chunk.page_content[:200])  # Mostrar los primeros 200 caracteres del chunk
    print()

Se han creado 24 chunks de texto para el embedding.
--- Chunk 1 (55 chars) ---
TENERIFE – LUGARES DE INTERÉS
SITIOS QUE VER
ZONA NORTE

--- Chunk 2 (976 chars) ---
• Santa Cruz de Tenerife:
Santa Cruz de Tenerife es la capital de la isla. Quizás la ruta a seguir si vais a Santa
Cruz sería:
- Aparcar en el aparcamiento del Parque Marítimo (ubicación).
- Caminar p

--- Chunk 3 (782 chars) ---
• San Cristóbal de La Laguna
Es ciudad Patrimonio de la Humanidad. Merece la pena un paseo por el centro,
repleto de edificios de estilo colonial. Quizás la ruta a seguir sería:
o Aparcar cerca de la 



Creamos una funcion para extraer los metadatos que nos interesan

In [4]:
from langchain_core.documents import Document
from typing import List

def clasificar_chunks_por_estructura(chunks_puros: List[Document]) -> List[Document]:
    """
    Recorre los chunks secuencialmente y les asigna metadatos de 'zona' y 'tipo'
    basándose en el contenido del texto (máquina de estados).
    """
    # Variables de estado iniciales
    zona_actual = "Desconocida"
    tipo_actual = "Desconocido"
    
    chunks_procesados = []
    
    for chunk in chunks_puros:
        # Convertimos a mayúsculas para que la búsqueda sea insensible a mayúsculas/minúsculas
        texto_clean = chunk.page_content.upper()
        
        # Detectar cambios de ZONA
        if "ZONA NORTE" in texto_clean:
            zona_actual = "Norte"
        elif "ZONA SUR" in texto_clean:
            zona_actual = "Sur"
            
        # Detectar cambios de TIPO
        if "SITIOS QUE VER" in texto_clean:
            tipo_actual = "Sitio de interes"
        elif "RESTAURANTES" in texto_clean:
            tipo_actual = "Restaurante"
            
        # Clonar las metadatos existentes del chunk (por si el PDF traía número de página, etc.)
        # y añadirle nuestras nuevas etiquetas de zona y tipo.
        nuevos_metadatos = chunk.metadata.copy()
        nuevos_metadatos.update({
            "zona": zona_actual,
            "tipo": tipo_actual
        })
        
        # Crear el nuevo documento enriquecido
        doc_etiquetado = Document(
            page_content=chunk.page_content,
            metadata=nuevos_metadatos
        )
        
        chunks_procesados.append(doc_etiquetado)
        
    return chunks_procesados

Y procesamos los chunks generados previamente para que tengan metadatos

In [5]:
chunks_listos_para_chroma = clasificar_chunks_por_estructura(chunks)

# # Comrobamos los chunks procesados
# for chunk in chunks_listos_para_chroma:
#     print(f"Zona: {chunk.metadata.get('zona', 'Desconocida')} | Tipo: {chunk.metadata.get('tipo', 'Desconocido')}")
#     print(chunk.page_content[:100])
#     print()


## 4. Indexación: embeddings y vector store
Creamos las incrustaciones con Gemini y guarda los chunks en una vectordb para poder hacer búsquedas por similaridad.

In [6]:
from langchain_core.embeddings import Embeddings

class GeminiEmbeddingFunction(Embeddings):
    """Adaptador entre la API de Gemini y ChromaDB — implementa la interfaz
    EmbeddingFunction para que ChromaDB pueda generar embeddings con Gemini 
    de forma transparente."""
    def __init__(self, api_client: genai.Client, model_name: str = EMBEDDING_MODEL):
        """
        Inicializamos el cliente de Google GenAI.
        """
        # El cliente busca automáticamente os.environ["GEMINI_API_KEY"]
        self.client = api_client
        self.model_name = model_name

    def embed_documents(self, texts: list[str]) -> list[list[float]]:
        """Método que LangChain llama para vectorizar los chunks"""
        response = self.client.models.embed_content(
            model=self.model_name,
            contents=texts
        )
        return [embedding.values for embedding in response.embeddings]

    def embed_query(self, text: str) -> list[float]:
        """Método que LangChain llama para vectorizar la pregunta del usuario"""
        response = self.client.models.embed_content(
            model=self.model_name,
            contents=text
        )
        return response.embeddings[0].values

Generamos el vector store

In [7]:
import shutil
from langchain_chroma import Chroma

embeddings = GeminiEmbeddingFunction(client, model_name=EMBEDDING_MODEL)

# Reseteamos la carpeta de persistencia para que no haya conflictos con datos antiguos
if CHROMA_PATH.exists():
    shutil.rmtree(CHROMA_PATH)

# Conpersistencia en disco usando Chroma
vector_store = Chroma.from_documents(
    documents=chunks_listos_para_chroma,
    embedding=embeddings,
    persist_directory=CHROMA_PATH
) 
print(f"Total de chunks en ChromaDB: {vector_store._collection.count()}")


Total de chunks en ChromaDB: 24


## 5. RAG: Recuperación de contexto
Búsqueda semántica en ChromaDB e inyección de contexto en el prompt

In [8]:
## Función para buscar los chunks relevantes en ChromaDB 
def buscar_chunks_relevantes(query: str, k: int = 3) -> str:
    """
    Busca los k chunks más relevantes en ChromaDB para la query dada.
    Devuelve un string concatenado de los resultados con metadatos de zona y tipo.
    """
    resultados = vector_store.similarity_search(query, k=k)
    
    # Unimos los fragmentos inyectando la zona y el tipo directamente antes del texto
    contexto_concatenado = "\n\n".join([f"[{res.metadata.get('zona')} - {res.metadata.get('tipo')}]: {res.page_content}" for res in resultados])
    
    return contexto_concatenado

In [9]:
## Función que construye la lista de mensajes: [SystemMessage] + historial + [HumanMessage(pregunta_actual)]
from langchain_core.messages import SystemMessage, HumanMessage, AIMessage

def construir_mensajes(pregunta_actual: str, contexto_rag: str, historial_conversacion: list[dict] = None) -> list[dict]:
    """
    Construye la lista de mensajes para el modelo de chat.
    [SystemMessage] + historial_conversacion + [HumanMessage(contexto + pregunta)]    """
    if historial_conversacion is None:
        historial_conversacion = []

    system_prompt = (
    "Eres un asistente virtual de viajes experto y amable para viajeros que quieren saber de Tenerife.\n"
    "Tu misión es responder a la pregunta del usuario utilizando ÚNICAMENTE el contexto proporcionado.\n"
    "CRÍTICO: Al final de tu respuesta cita las fuentes usadas. Ejemplo: 'Fuentes: [Sur - Sitio de interes], [Norte - Restaurante]'\n"
    "Si en el contexto no encuentras la respuesta, di amablemente que no dispones de esa información.\n\n"
    )
    
    # Iniciamos la lista con el SystemMessage
    mensajes = [SystemMessage(content=system_prompt)]
    # Sumamos el historial de conversación
    mensajes += historial_conversacion
    
    # Construimos el ÚLTIMO HumanMessage empaquetando el contexto RAG y la pregunta actual
    contenido_humano_final = (
        f"CONTEXTO:\n{contexto_rag}\n\n"f"Pregunta: {pregunta_actual}"
    )

    # Añadimos el mensaje final a la lista
    mensajes += [HumanMessage(content=contenido_humano_final)]

    return mensajes

## 6. Diálogo multiturno
Gestión del historial de conversación con control de ventana de tokens

In [10]:
def controlar_ventana_tokens(historial_conversacion: list, max_turnos: int = 3) -> list:
    """
    Controla la ventana de tokens del modelo limitando el historial por turnos.
    Garantiza la coherencia eliminando siempre pares completos (HumanMessage + AIMessage)
    desde el inicio del hilo de la conversación.
    """
    # Cada turno de chat (pregunta + respuesta) equivale a 2 mensajes individuales
    max_mensajes = max_turnos * 2
    
    # Si el volumen de mensajes supera el límite de la ventana, recortamos por el principio
    while len(historial_conversacion) > max_mensajes:
        historial_conversacion.pop(0)  # Elimina el HumanMessage más antiguo
        historial_conversacion.pop(0)  # Elimina el AIMessage más antiguo
        
    return historial_conversacion

In [11]:
from langchain.chat_models import init_chat_model
from langchain_core.language_models import BaseChatModel

def chat(pregunta: str, langchain_model: BaseChatModel ) -> str:
    """Función principal que ejecuta el flujo de los 5 pasos."""
    global historial
    
    # Aplicamos la ventana de tokens antes de comenzar la logica
    historial = controlar_ventana_tokens(historial)

    # Obtener contexto formateado con citaciones
    contexto = buscar_chunks_relevantes(pregunta, k=3)
    
    # Construir la lista de mensajes ([System] + historial + [Human con Contexto])
    lista_mensajes = construir_mensajes(
        pregunta_actual=pregunta, 
        contexto_rag=contexto, 
        historial_conversacion=historial
    )
    
    # Llamar al modelo
    respuesta_llm = langchain_model.invoke(lista_mensajes)
    texto_respuesta = respuesta_llm.content
    
    # Añadir la pregunta original (limpia) y la respuesta al historial persistente
    # Nota: Guardamos la pregunta limpia en el historial para no contaminar los turnos futuros con contextos viejos
    historial.append(HumanMessage(content=pregunta))
    historial.append(AIMessage(content=texto_respuesta))
    
    return texto_respuesta

In [12]:
historial = []
langchain_model = init_chat_model(GENERATION_MODEL, model_provider="google_genai")

In [ ]:

print(chat("¿Qué sitios recomiendas visitar en el norte de Tenerife?", langchain_model))
print(chat("¿Cuál de esos sitios tiene más historia?", langchain_model))
print(chat("¿Hay buenos restaurantes en la misma zona?", langchain_model))
print(chat("¿Y qué tiempo suele hacer por esa zona?", langchain_model))

In [ ]:
print(chat("¿Qué playas o piscinas hay en el Norte de Tenerife?", langchain_model))
print(chat("¿Y restaurantes cerca de alguna de ellas?", langchain_model))

## 7. Herramienta externa: predicción meteorológica
Integración con Open-Meteo API para consultar el tiempo real en Tenerife
Function calling con get_weather

In [13]:
import requests

def api_openmeteo(latitude: float, longitude: float) -> dict:
    """
    Consulta la API de Open-Meteo y devuelve el pronóstico de 7 días 
    estructurado por fechas en formato JSON.
    
    NOTA PARA EL MODELO: Si la respuesta contiene la clave 'error', 
    informa amablemente al usuario de que el servicio externo no está disponible.
    """

    url = "https://api.open-meteo.com/v1/forecast"
    params = {
        "latitude": latitude,
        "longitude": longitude,
        "hourly": ["temperature_2m", "precipitation_probability"],
        "timezone": "Atlantic/Canary"  # Forzamos la hora local de Tenerife
    }

    try:
        # Petición directa y verificación de errores HTTP (ej: 404, 500)
        response = requests.get(url, params=params)
        response.raise_for_status()

        # Parseamos a JSON puro
        data = response.json()
        
        # Extraemos las listas del JSON
        lista_horas = data["hourly"]["time"]                     # Lista de strings: ["2026-06-15T00:00", ...]
        lista_temps = data["hourly"]["temperature_2m"]           # Lista de floats
        lista_lluvia = data["hourly"]["precipitation_probability"] # Lista de ints
        
        # Agrupamos los datos por día usando un diccionario nativo
        pronostico_semanal = {}

        # Separamos por la 'T' para agrupar los datos horarios bajo una misma clave de fecha (YYYY-MM-DD)
        for hora, temp, lluvia in zip(lista_horas, lista_temps, lista_lluvia):
            fecha_dia = hora.split("T")[0]

            # Si es la primera vez que vemos este día, inicializamos su estructura
            if fecha_dia not in pronostico_semanal:
                pronostico_semanal[fecha_dia] = {
                    "temperaturas": [],
                    "probabilidades_lluvia": []
                }
            
            # Acumulamos los valores de cada hora de ese día
            pronostico_semanal[fecha_dia]["temperaturas"].append(temp)
            pronostico_semanal[fecha_dia]["probabilidades_lluvia"].append(lluvia)

        # Reducimos los datos a las métricas clave (Max, Min, Lluvia) para ahorrar tokens
        resumen_final = {}
        for fecha_dia, datos in pronostico_semanal.items():
            temp_max = max(datos["temperaturas"])
            temp_min = min(datos["temperaturas"])
            lluvia_max = max(datos["probabilidades_lluvia"])
            
            # Reducimos las 24 horas a métricas clave del día para optimizar la ventana de 
            # contexto del LLM y ahorrar tokens
            resumen_final[fecha_dia] = {
                "resumen_del_dia": f"Max: {temp_max:.1f}°C, Min: {temp_min:.1f}°C, Lluvia máx: {lluvia_max}%"
            }
                
        return {
            "unidad_temperatura": "°C",
            "unidad_precipitacion": "%",
            "pronostico_7_dias": resumen_final
            }
    except Exception as e:
        return {"error": f"Fallo al conectar con el servicio meteorológico: {str(e)}"}
         

In [14]:
from langchain_core.tools import tool

@tool
def get_weather(zone: str) -> dict:
    """
    Obtiene el pronóstico meteorológico detallado para los próximos 7 días 
    en una zona específica de la isla de Tenerife.

    Esta función actúa como una herramienta para el modelo de lenguaje (LLM). 
    Debe ser invocada siempre que el usuario pregunte por el clima actual, 
    la predicción para mañana, las condiciones generales de la semana o si 
    busca recomendaciones de actividades basadas en el tiempo en Tenerife.

    Args:
        zona (str): Región geográfica de Tenerife para la consulta. 
            Debe ser estrictamente uno de los siguientes valores:
            - "Norte": Puerto de la Cruz, La Orotava, Icod, etc.
            - "Sur": Los Cristianos, Las Américas, Costa Adeje, etc.
    Returns:
        dict: Un diccionario JSON con el pronóstico de 7 días. Cada clave es una 
        fecha (DD-MM-YYYY) y su valor contiene un resumen con temperaturas (Max/Min) 
        y la probabilidad máxima de precipitación.
        
        Si ocurre un error de red o la zona no es válida, devuelve un diccionario 
        con la clave 'error' y la descripción del fallo.

    Instrucciones para el LLM:
        1. Analiza el JSON devuelto y mapea de forma autónoma la fecha que el 
           usuario solicita (hoy, mañana, fin de semana) con las claves del diccionario.
        2. Si la respuesta contiene la clave 'error', no inventes datos climáticos; 
           informa al usuario con amabilidad de que el servicio meteorológico no está disponible.
    """
    coordenadas = {
    "norte": {"lat": 28.39, "lon": -16.52},
    "sur": {"lat": 28.05, "lon": -16.71},
    }
    if zone.lower() in coordenadas:
        lat = coordenadas[zone.lower()]["lat"]
        lon = coordenadas[zone.lower()]["lon"]
    else:
         return { "error": f"La zona '{zone}' no está registrada. "
                   f"Las zonas válidas son: {list(coordenadas.keys())}." }
        

    resultado_clima = api_openmeteo(
        latitude=lat, 
        longitude=lon
        )
    
    return resultado_clima
    

## 8. Agente conversacional con function calling
Loop de llamadas: el LLM decide cuándo invocar herramientas externa

In [37]:
from langchain_core.messages import HumanMessage, AIMessage, ToolMessage

def chat_with_tools(pregunta: str, langchain_model_con_tools) -> str:
    """
    Función principal adaptada para soportar Tool Calling de forma reactiva.
    """
    global historial
    
    # Control de la ventana de tokens (Mantiene el historial limpio antes de procesar)
    historial = controlar_ventana_tokens(historial, max_turnos=2)

    # Obtener contexto del RAG por si la pregunta es sobre información general
    contexto = buscar_chunks_relevantes(pregunta, k=3)
    
    # Construir la lista de mensajes con la estructura correcta
    lista_mensajes = construir_mensajes(
        pregunta_actual=pregunta, 
        contexto_rag=contexto, 
        historial_conversacion=historial
    )
    
    # Primera llamada al modelo (el modelo decidirá si responde o usa la Tool)
    respuesta_llm = langchain_model_con_tools.invoke(lista_mensajes)

    # FLUJO CON TOOL CALLING: ¿El modelo ha solicitado usar una herramienta
    texto_respuesta = ""
    if respuesta_llm.tool_calls:
        # Registro de intentos 
        logger.info(f" [TOOL CALL DETECTADA] El LLM llama a la herramienta 'get_weather'")
        
        # El modelo no devuelve texto directo, devuelve una petición de ejecución
        for tool_call in respuesta_llm.tool_calls:
            if tool_call["name"] == "get_weather":
                # Ejecutamos la función real en Python con los argumentos que extrajo Gemini
                argumentos = tool_call["args"]
                resultado_clima = get_weather.invoke({"zone": argumentos.get("zone")})                
                # Para que el flujo continúe, debemos añadir la petición del modelo 
                # y la respuesta de la herramienta a la lista de mensajes
                lista_mensajes.append(respuesta_llm) 
                lista_mensajes.append(
                    ToolMessage(
                        content=str(resultado_clima), 
                        tool_call_id=tool_call["id"]
                    )
                )
                
                # Segunda llamada al modelo: Ahora Gemini lee los datos de Open-Meteo y redacta la respuesta final
                segunda_respuesta = langchain_model_con_tools.invoke(lista_mensajes)
                content = segunda_respuesta.content
                texto_respuesta = content[0]["text"] if isinstance(content, list) else content
                if "[get_weather]" not in texto_respuesta:
                    texto_respuesta += "\n\nFuentes: [get_weather]"

    else:
        # FLUJO CORRIENTE: El modelo respondió directamente usando solo el contexto del RAG
        content = respuesta_llm.content
        texto_respuesta = content[0]["text"] if isinstance(content, list) else content
    

    # Guardamos la pregunta limpia y la respuesta final (ocultando la Tool)
    # para que los próximos turnos mantengan una coherencia de chat perfecta.
    historial.append(HumanMessage(content=pregunta))
    historial.append(AIMessage(content=texto_respuesta))
    
    return texto_respuesta

In [16]:
historial = []
langchain_model = init_chat_model(GENERATION_MODEL, model_provider="google_genai")
langchain_model_con_tools = langchain_model.bind_tools([get_weather])


In [ ]:

print(chat_with_tools("¿Qué playas o piscinas hay en el Norte de Tenerife?", langchain_model_con_tools))
print(chat_with_tools("¿Es buen momento para ir a la playa en el norte esta semana?", langchain_model_con_tools))
print(chat_with_tools("A que restaurante voy si estoy por esa zona?", langchain_model_con_tools))


## 9. Evaluacion y métricas
Evaluación del asistente: latencia, citación de fuentes y detección de tool calling sobre un conjunto fijo de prompts

In [45]:
import time
def evaluar_llm(prompt, modelo_con_tools):
    """
    Ejecuta un prompt y calcula métricas de rendimiento y contenido.
    """
    # Medir tiempo de respuesta
    inicio = time.time()
    global historial
    historial = []
    try:        
        respuesta = chat_with_tools(prompt, modelo_con_tools)
        tiempo_respuesta = time.time() - inicio
    except Exception as e:
        return {
            "prompt": prompt,
            "error": str(e),
            "longitud_caracteres": 0,
            "contiene_fuentes": False,
            "uso_tool_calling": False,
            "tiempo_segundos": time.time() - inicio
        }

    # Extraemos el texto del contenido y calculamos longitud
    longitud_caracteres = len(respuesta)

    # Nota: Para tokens exactos necesitarías el tiktoken del modelo, 
    # pero si el modelo devuelve metadata, a veces viene aquí:
    # tokens = respuesta.response_metadata.get("token_usage", {}).get("completion_tokens")

    # Verificamos si citó fuentes
    contiene_fuentes = "Fuentes:" in respuesta

    # Verificamos Tool Calling: detectar por el texto, igual que contiene_fuentes
    uso_tool_calling = "get_weather" in respuesta
            
    return {
        "prompt": prompt,
        "longitud_caracteres": longitud_caracteres,
        "contiene_fuentes": contiene_fuentes,
        "uso_tool_calling": uso_tool_calling,
        "tiempo_segundos": round(tiempo_respuesta, 3),
        "respuesta": respuesta
    }

In [51]:
import pandas as pd

PROMPTS_EVALUACION = [
    # RAG - Zona Norte
    "¿Qué lugares recomiendas visitar en el norte de Tenerife?",
    "¿Qué restaurantes hay en la zona norte?",
    
    # RAG - Zona Sur  
    "¿Qué hay que ver en Costa Adeje?",
    "¿Cuál es el mejor parque acuático de Tenerife?",
    
    # Tool calling
    "¿Qué tiempo va a hacer esta semana en el norte de Tenerife?",
    "¿Lloverá en el sur de Tenerife los próximos días?",
    
    # Pregunta sin respuesta en el PDF
    "¿Cuánto cuesta la entrada al Loro Parque?",
    
    # Pregunta ambigua
    "¿Qué me recomiendas hacer mañana?",
    
    # Multiturno implícito
    "¿Y en el sur, qué playas hay?",
]
GROUND_TRUTH = {
    "¿Qué lugares recomiendas visitar en el norte de Tenerife?": {
        "contiene_fuentes": True,
        "uso_tool_calling": False,
    },
    "¿Qué restaurantes hay en la zona norte?": {
        "contiene_fuentes": True,
        "uso_tool_calling": False,
    },
    "¿Qué tiempo va a hacer esta semana en el norte de Tenerife?": {
        "contiene_fuentes": True,
        "uso_tool_calling": True,
    },
    "¿Lloverá en el sur de Tenerife los próximos días?": {
        "contiene_fuentes": True,
        "uso_tool_calling": True,
    },
    "¿Cuánto cuesta la entrada al Loro Parque?": {
        "contiene_fuentes": False,
        "uso_tool_calling": False,
    },
    "¿Qué me recomiendas hacer mañana?": {
        "contiene_fuentes": False,
        "uso_tool_calling": False,
    },
    "¿Y en el sur, qué playas hay?": {
        "contiene_fuentes": True,
        "uso_tool_calling": False,
    }
}

resultados = []

for p in GROUND_TRUTH.keys():
    print(f"Evaluando: '{p}'...")
    metricas = evaluar_llm(p, langchain_model_con_tools)
    resultados.append(metricas)

    # Obtenemos lo que esperábamos (Ground Truth)
    esperado = GROUND_TRUTH[p]
    
    # Comparamos Real vs Esperado (Generar métricas de acierto)
    acierto_tool = (metricas["uso_tool_calling"] == esperado["uso_tool_calling"])
    acierto_fuentes = (metricas["contiene_fuentes"] == esperado["contiene_fuentes"])
    
    # 4. Consolidar todo en un único registro
    registro = {
        "prompt": p,
        "tiempo_segundos": metricas["tiempo_segundos"],
        "longitud_caracteres": metricas["longitud_caracteres"],
        # Datos de Tool Calling
        "tool_real": metricas["uso_tool_calling"],
        "tool_esperado": esperado["uso_tool_calling"],
        "tool_correcto": acierto_tool,
        # Datos de Fuentes
        "fuentes_real": metricas["contiene_fuentes"],
        "fuentes_esperado": esperado["contiene_fuentes"],
        "fuentes_correcto": acierto_fuentes,
    }

    resultados.append(registro)

df_resultados = pd.DataFrame(resultados)


Evaluando: '¿Qué lugares recomiendas visitar en el norte de Tenerife?'...
Evaluando: '¿Qué restaurantes hay en la zona norte?'...
Evaluando: '¿Qué tiempo va a hacer esta semana en el norte de Tenerife?'...
Evaluando: '¿Lloverá en el sur de Tenerife los próximos días?'...
Evaluando: '¿Cuánto cuesta la entrada al Loro Parque?'...
Evaluando: '¿Qué me recomiendas hacer mañana?'...
Evaluando: '¿Y en el sur, qué playas hay?'...


In [52]:
import plotly.graph_objects as go
from plotly.subplots import make_subplots

# Limpiamos la variable
fig = go.Figure()
# Creamos una figura con 4 subplots (2 filas, 2 columnas)
fig = make_subplots(
    rows=2, cols=2,
    specs=[
        [{"type": "xy"},     {"type": "domain"}],
        [{"type": "xy"},    {"type": "domain"}]
    ],
    subplot_titles=(
        "Tiempo de Respuesta por Prompt", 
        "Presencia de Fuentes",
        "Longitud del Texto (Caracteres)", 
        "Uso de Tool Calling"
    ),
    vertical_spacing=0.15,
    horizontal_spacing=0.12
)

# Gráfico 1: Tiempos de Respuesta (Barras horizontales)
fig.add_trace(
    go.Bar(
        y=df_resultados["prompt"],
        x=df_resultados["tiempo_segundos"],
        orientation='h',
        name="Tiempo (s)",
        marker=dict(color='rgb(55, 83, 109)')
    ),
    row=1, col=1
)

# Gráfico 2: Longitud de caracteres (Barras horizontales) 
fig.add_trace(
    go.Bar(
        y=df_resultados["prompt"],
        x=df_resultados["longitud_caracteres"],
        orientation='h',
        name="Caracteres",
        marker=dict(color='rgb(26, 118, 255)')
    ),
    row=2, col=1
)

# Gráfico 3: Distribución de Tool Calling (Pastel)
tool_counts = df_resultados["uso_tool_calling"].value_counts()
fig.add_trace(
    go.Pie(
        labels=tool_counts.index.map({True: 'Usó Tool', False: 'No usó Tool'}),
        values=tool_counts.values,
        name="Tool Calling",
        hole=0.3
    ),
    row=2, col=2
)

# Gráfico 4: Distribución de Fuentes citadas (Pastel)
fuentes_counts = df_resultados["contiene_fuentes"].value_counts()
fig.add_trace(
    go.Pie(
        labels=fuentes_counts.index.map({True: 'Con Fuentes', False: 'Sin Fuentes'}),
        values=fuentes_counts.values,
        name="Fuentes Citadas",
        hole=0.3
    ),
    row=1, col=2
)

# Personalizar el diseño general (Layout)
fig.update_layout(
    title_text="Dashboard de Evaluación de Prompts LLM",
    title_font_size=22,
    height=700, 
    width=1000,
    showlegend=False, # Ocultamos leyenda global para evitar desorden
    template="plotly_white"
)

# Ajustar ejes específicos para los gráficos de barras
fig.update_xaxes(title_text="Segundos", row=1, col=1)
fig.update_xaxes(title_text="Cantidad de caracteres", row=1, col=2)
fig.update_yaxes(autorange="reversed", row=1, col=1) # Para que el primer prompt aparezca arriba
fig.update_yaxes(autorange="reversed", row=1, col=2)

# Mostrar el gráfico
# Si estás en Jupyter Notebook/Lab se mostrará automáticamente:
fig.show()

# (Opcional) Si prefieres guardarlo como un archivo HTML interactivo:
# fig.write_html("evaluacion_prompts.html")

## Analizar los resultados globales

In [54]:
# Calcular el porcentaje de precisión (Accuracy)
precision_tool = df_resultados["tool_correcto"].mean() * 100
precision_fuentes = df_resultados["fuentes_correcto"].mean() * 100

print(f"\n=== Reporte de Calidad vs Ground Truth ===")
print(f"Precisión en Tool Calling: {precision_tool:.2f}%")
print(f"Precisión en Citar Fuentes: {precision_fuentes:.2f}%")

# Ver qué prompts fallaron
fallas_tool = df_resultados[df_resultados["tool_correcto"] == False]
if not fallas_tool.empty:
    print("\nPrompts donde falló el Tool Calling:")
    print(fallas_tool[["prompt", "tool_real", "tool_esperado"]])


=== Reporte de Calidad vs Ground Truth ===
Precisión en Tool Calling: 100.00%
Precisión en Citar Fuentes: 100.00%
